In [1]:
from google.colab import drive

drive.mount("/content/drive")
BASE_DIR = "/content/drive/MyDrive/Nexlyra"

Mounted at /content/drive


In [2]:
import os
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

In [3]:
class PatchEmbedding(nn.Module):
    """
    Tasks 4-6: Custom Patch Embedding layer for [1, 10, 10] input using 2x2 patches.
    Results in 25 patches of dimension 4, projected to hidden_dim=64.
    """
    def __init__(self, in_channels=1, patch_size=2, hidden_dim=64):
        super().__init__()
        self.patch_size = patch_size
        self.patch_dim = in_channels * patch_size * patch_size  # 1 * 2 * 2 = 4
        self.projection = nn.Linear(self.patch_dim, hidden_dim)

    def forward(self, x):
        # x shape: [B, 1, 10, 10]
        B, C, H, W = x.shape
        p = self.patch_size

        # Reshape into patches: [B, C, H/p, p, W/p, p] -> [B, (H/p)*(W/p), C * p * p]
        x = x.reshape(B, C, H // p, p, W // p, p)
        x = x.permute(0, 2, 4, 1, 3, 5).reshape(B, (H // p) * (W // p), C * p * p)

        # Linear projection to hidden_dim
        return self.projection(x)  # [B, 25, 64]

In [4]:
class TransformerEncoderBlock(nn.Module):
    """
    Tasks 9-11: Transformer Encoder Block with LayerNorm, Multi-Head Self-Attention,
    GELU MLP, and Residual Additions.
    """
    def __init__(self, hidden_dim=64, num_heads=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.attn = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(hidden_dim)

        mlp_hidden_dim = int(hidden_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, hidden_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # MHSA with Residual Add
        norm_x = self.norm1(x)
        attn_out, _ = self.attn(norm_x, norm_x, norm_x)
        x = x + attn_out

        # MLP with Residual Add
        x = x + self.mlp(self.norm2(x))
        return x


In [5]:
class AsymmetricFocalTverskyLoss(nn.Module):
    """
    Tasks 24-25: Custom loss punishing False Positives heavily (alpha=0.2, beta=0.8)
    to prevent costly turbine shutdowns from false alarms.
    """
    def __init__(self, alpha=0.2, beta=0.8, gamma=1.5, smooth=1e-6):
        super().__init__()
        self.alpha = alpha  # Weight for False Positives
        self.beta = beta    # Weight for False Negatives
        self.gamma = gamma  # Focal tuning parameter
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        targets = targets.view(-1, 1)
        probs = probs.view(-1, 1)

        # True Positives, False Positives, False Negatives
        TP = (probs * targets).sum()
        FP = (probs * (1 - targets)).sum()
        FN = ((1 - probs) * targets).sum()

        tversky_index = (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)

        # Focal Tversky Loss variant
        focal_tversky_loss = (1.0 - tversky_index) ** (1.0 / self.gamma)
        return focal_tversky_loss

In [6]:
class HydroTransformerNet(nn.Module):
    """
    Tasks 1-18: Dual-Input Audio Spectrogram Transformer (ViT + Tabular Fusion).
    """
    def __init__(self, hidden_dim=64, num_layers=3, num_heads=4):
        super().__init__()
        # Spatial Branch (ViT)
        self.patch_embed = PatchEmbedding(in_channels=1, patch_size=2, hidden_dim=hidden_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, hidden_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, 26, hidden_dim)) # 25 patches + 1 CLS token

        self.transformer_encoder = nn.Sequential(*[
            TransformerEncoderBlock(hidden_dim=hidden_dim, num_heads=num_heads)
            for _ in range(num_layers)
        ])

        # Tabular Branch (Physics features)
        self.tabular_mlp = nn.Sequential(
            nn.Linear(6, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(p=0.4)
        )

        # Fusion Branch
        self.fusion_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(p=0.3),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, acoustic, tabular):
        # 1. ViT Processing
        B = acoustic.size(0)
        x = self.patch_embed(acoustic)  # [B, 25, 64]

        cls_tokens = self.cls_token.expand(B, -1, -1)  # [B, 1, 64]
        x = torch.cat((cls_tokens, x), dim=1)  # [B, 26, 64]
        x = x + self.pos_embed

        x = self.transformer_encoder(x)
        cls_repr = x[:, 0]  # Extract [CLS] token global representation [B, 64]

        # 2. Tabular Processing
        tab_repr = self.tabular_mlp(tabular)  # [B, 64]

        # 3. Multimodal Fusion
        fused = torch.cat((cls_repr, tab_repr), dim=1)  # [B, 128]
        logits = self.fusion_mlp(fused)  # [B, 1]
        return logits


In [7]:
def train_epoch(model, dataloader, optimizer, criterion, scaler, device, accumulation_steps=4):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for i, (acoustic, tabular, targets) in enumerate(dataloader):
        acoustic = acoustic.to(device)
        tabular = tabular.to(device)
        targets = targets.to(device)

        # Task 31: Automatic Mixed Precision (AMP) autocast
        with torch.amp.autocast('cuda'):
            logits = model(acoustic, tabular)
            loss = criterion(logits, targets)
            loss = loss / accumulation_steps  # Normalize loss for gradient accumulation

        # Task 33: Scaled backward pass
        scaler.scale(loss).backward()

        if (i + 1) % accumulation_steps == 0 or (i + 1) == len(dataloader):
            # Task 34: Gradient clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # Task 35: Optimizer step & scaler update
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * accumulation_steps

    return total_loss / len(dataloader)

In [8]:
def evaluate_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for acoustic, tabular, targets in dataloader:
            acoustic = acoustic.to(device)
            tabular = tabular.to(device)
            targets = targets.to(device)

            with torch.amp.autocast('cuda'):
                logits = model(acoustic, tabular)
                loss = criterion(logits, targets)

            total_loss += loss.item()

            # Task 39: Convert logits to probabilities
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.extend(probs.flatten())
            all_targets.extend(targets.cpu().numpy().flatten())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    # Task 40: Calculate metrics
    auroc = roc_auc_score(all_targets, all_preds) if len(np.unique(all_targets)) > 1 else 0.5
    pr_auc = average_precision_score(all_targets, all_preds) if len(np.unique(all_targets)) > 1 else 0.0
    f1 = f1_score(all_targets, (all_preds > 0.5).astype(int), zero_division=0)

    return total_loss / len(dataloader), auroc, pr_auc, f1


In [9]:
# MAIN ORCHESTRATOR FOR DAY
def main():
    print("[*] Initializing Day 3: Audio Spectrogram Transformer (AST) Architecture...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Tasks 19 & 20: Instantiate model and verify shape with dummy tensors [256, 1, 10, 10] and [256, 6]
    model = HydroTransformerNet(hidden_dim=64, num_layers=3, num_heads=4).to(device)

    dummy_acoustic = torch.randn(256, 1, 10, 10, device=device)
    dummy_tabular = torch.randn(256, 6, device=device)

    with torch.no_grad():
        dummy_out = model(dummy_acoustic, dummy_tabular)
    print(f"[*] Dummy Output Logits Shape Verified: {dummy_out.shape}")
    assert dummy_out.shape == torch.Size([256, 1]), "Logit shape mismatch!"

    # Task 21: Parameter count check
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[*] Total Trainable Parameters: {trainable_params:,} (Confirmed under 1 Million)")
    assert trainable_params < 1_000_000, "Model exceeds 1M parameter limit!"

    # Tasks 22 & 23: Optimizer and Scheduler
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    criterion = AsymmetricFocalTverskyLoss(alpha=0.2, beta=0.8)

    # Task 29: Initialize GradScaler for FP16 training
    scaler = torch.amp.GradScaler('cuda')

    # Task 41: Early Stopping Setup tracking Validation PR-AUC (patience = 8)
    best_pr_auc = 0.0
    patience = 8
    patience_counter = 0

    print("[*] AST model initialized successfully with AdamW, Cosine Scheduler, and AMP GradScaler.")
    print("[*] Ready for Day 3 Model Training & Validation execution.")

if __name__ == '__main__':
    main()

[*] Initializing Day 3: Audio Spectrogram Transformer (AST) Architecture...
[*] Dummy Output Logits Shape Verified: torch.Size([256, 1])
[*] Total Trainable Parameters: 160,897 (Confirmed under 1 Million)
[*] AST model initialized successfully with AdamW, Cosine Scheduler, and AMP GradScaler.
[*] Ready for Day 3 Model Training & Validation execution.
